## Extract the data

In [14]:
from nba_api.stats.endpoints import leagueleaders
import pandas as pd

# Considering 2025-26 Regular Season
leaders = leagueleaders.LeagueLeaders(
    season='2025-26',                     
    season_type_all_star='Regular Season',
    per_mode48='Totals',                  # 'Totals', 'PerGame', 'Per36', 'Per48', etc.
    stat_category_abbreviation='PTS',     # (PTS, REB, AST...), only orders by this stat
    scope='S',                            # 'S' = all players, 'R' = rookies
    league_id='00'                        # NBA
)

# Get data as DataFrame
df = leaders.get_data_frames()[0]
print(df.head())

   PLAYER_ID  RANK                   PLAYER     TEAM_ID TEAM  GP   MIN  FGM  \
0    1628983     1  Shai Gilgeous-Alexander  1610612760  OKC  48  1605  526   
1    1628378     2         Donovan Mitchell  1610612739  CLE  47  1592  471   
2    1629029     3              Luka Dončić  1610612747  LAL  40  1447  426   
3    1630178     4             Tyrese Maxey  1610612755  PHI  46  1796  474   
4    1627759     5             Jaylen Brown  1610612738  BOS  45  1543  488   

    FGA  FG_PCT  ...  REB  AST  STL  BLK  TOV   PF   PTS   EFF  AST_TOV  \
0   942   0.558  ...  213  305   62   38  100   99  1538  1592     3.05   
1   975   0.483  ...  218  274   69   11  147  115  1352  1229     1.86   
2   895   0.476  ...  319  353   60   19  169  101  1345  1360     2.09   
3  1005   0.472  ...  193  317   90   40  112   98  1343  1309     2.83   
4  1006   0.485  ...  312  216   47   18  162  126  1323  1164     1.33   

   STL_TOV  
0     0.62  
1     0.47  
2     0.36  
3     0.80  
4     0.2

## Analysis

## Load to BQ

In [47]:
!pip install pandas-gbq google-cloud-bigquery


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Users\Felo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from pandas_gbq import to_gbq

to_gbq(
    dataframe=df,
    destination_table="leagueleaders.season_25_26",
    project_id="nba-stats-485814",
    if_exists="replace"  
)


100%|██████████| 1/1 [00:00<?, ?it/s]


## Queries

In [ ]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd

credentials_path='complete path'

client = bigquery.Client(
      credentials=service_account.Credentials.from_service_account_file(
            credentials_path,
            scopes=["https://www.googleapis.com/auth/cloud-platform"],
        ),
      project='project-id'
)
        

def query(sql):
        try:
            query_job = client.query(sql)
            return query_job.to_dataframe()
        except Exception as e:
            print(f"Error: {e}")
            return pd.DataFrame()
    
def get_player_by_name(project_id, player_name, dataset_id, table_id):
        sql = f"""
        SELECT *
        FROM `{project_id}.{dataset_id}.{table_id}`
        WHERE UPPER(PLAYER) LIKE UPPER('%{player_name}%')
        ORDER BY PTS DESC
        LIMIT 20
        """
        return query(sql)

In [6]:
get_player_by_name(project_id = 'nba-stats-485814', 
                   player_name = 'Tyrese Maxey', 
                   dataset_id = 'leagueleaders', 
                   table_id = 'season_25_26')

C:\Users\Felo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,PLAYER_ID,RANK,PLAYER,TEAM_ID,TEAM,GP,MIN,FGM,FGA,FG_PCT,...,REB,AST,STL,BLK,TOV,PF,PTS,EFF,AST_TOV,STL_TOV
0,1630178,3,Tyrese Maxey,1610612755,PHI,45,1761,467,987,0.473,...,188,309,89,40,111,95,1325,1289,2.78,0.8


In [9]:
sql = """
WITH ranked AS (
  SELECT
    PLAYER_ID,
    PLAYER,
    TEAM,
    FG_PCT,
    FG3_PCT,
    FT_PCT,
    
    PERCENT_RANK() OVER (ORDER BY FG_PCT)   AS fg_pct_pr,
    PERCENT_RANK() OVER (ORDER BY FG3_PCT)  AS fg3_pct_pr,
    PERCENT_RANK() OVER (ORDER BY FT_PCT)   AS ft_pct_pr
    
  FROM `nba-stats-485814.leagueleaders.season_25_26`
  WHERE FG_PCT  IS NOT NULL
    AND FG3_PCT IS NOT NULL
    AND FT_PCT  IS NOT NULL
)

SELECT
  PLAYER_ID,
  PLAYER,
  TEAM,
  
  FG_PCT,
  ROUND(fg_pct_pr  * 100, 1) AS fg_pct_percentile,
  
  FG3_PCT,
  ROUND(fg3_pct_pr * 100, 1) AS fg3_pct_percentile,
  
  FT_PCT,
  ROUND(ft_pct_pr  * 100, 1) AS ft_pct_percentile

FROM ranked
WHERE PLAYER_ID = 1630178
"""

In [10]:
query(sql)

C:\Users\Felo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,PLAYER_ID,PLAYER,TEAM,FG_PCT,fg_pct_percentile,FG3_PCT,fg3_pct_percentile,FT_PCT,ft_pct_percentile
0,1630178,Tyrese Maxey,PHI,0.473,61.9,0.387,77.3,0.885,86.4
